# Build Your First Intelligent Agent Team: A Step-by-Step Guide to Multi-Tool and Multi-Agent Design with GCP ADK

In this tutorial, you’ll progressively build an intelligent, modular system using the **Google Cloud Agent Development Kit (ADK)**. Starting from a simple agent with a single tool, you’ll evolve your solution into a system of specialized agents that collaborate to handle a range of user tasks through **tool invocation** and **agent delegation**.

---

### ✅ Step 1: Single Agent, Single Tool — Weather Lookup

You’ll begin by creating your first agent: a focused assistant that retrieves weather information using a single tool. This foundational step introduces how tools provide capabilities and how agents reason about when to use them.

---

### ✅ Step 2: Single Agent, Multiple Tools — Weather + Event Recommendations

Next, you’ll enhance your agent by adding an additional tool for event recommendations. The agent now interprets user intent and chooses between tools—demonstrating **multi-tool reasoning** within a single decision-making entity.

---

### ✅ Step 3: Multi-Agent, Multi-Tool System — Specialized Agents + Delegation

In the final step, you'll refactor your system to include multiple agents, each specialized in a narrow task (e.g., greetings, farewells, weather, events). A root agent will serve as the coordinator, delegating tasks based on user intent and using the appropriate agent or tool to respond.

---

## 🔀 Multi-Tool vs Multi-Agent: Key Differences and Design Guidance

Choosing between multi-tool and multi-agent patterns depends on the scope, complexity, and maintainability of your use case. Here's how to decide:

| Pattern         | Description | When to Use |
|----------------|-------------|-------------|
| **Multi-Tool Reasoning** | A single agent chooses from multiple tools to fulfill user requests. | Use this when all capabilities are closely related, the logic is lightweight, or when a centralized design is easier to manage. |
| **Multi-Agent Delegation** | A root agent delegates to sub-agents, each specialized in a specific task. | Use this when responsibilities are clearly separable, when modularity is important, or when individual agents require different instructions, models, or scaling behavior. |


---

### 🛠 By the end of this tutorial, you’ll be able to:

- Define tools that give agents specific capabilities  
- Build agents that reason across tools to fulfill different intents  
- Design agent teams that collaborate through delegation and specialization  

Let’s get started.

---


##  Step 0: Setup, Installation and configuration

<pre><code>pip install google-adk</code></pre>


In [1]:
import os
import asyncio
from google.adk.agents import Agent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types


If you are running your code with GCP the use this part of code and make sure you update the **PROJECT_ID**

In [2]:
# GCP configuration
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = "vertex-ai-search-v2"#--PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = 'us-central1'




If you are running your code with GCP the use this part of code and make sure you update the **GOOGLE_API_KEY**

In [4]:
# API configuration

# MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

# os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
# os.environ["GOOGLE_API_KEY"] = --GOOGLE_API_KEY


## Step 1: Your First Agent \- Basic Weather Lookup (Single agent, single tool)

Let's begin by building the fundamental component of our Weather Bot: a single agent capable of performing a specific task – looking up weather information. This involves creating two core pieces:

1. **A Tool:** A Python function that equips the agent with the *ability* to fetch weather data.  
2. **An Agent:** The AI "brain" that understands the user's request, knows it has a weather tool, and decides when and how to use it.

---

**1\. Define the Tool (`get_weather`)**

In ADK, **Tools** are the building blocks that give agents concrete capabilities beyond just text generation. They are typically regular Python functions that perform specific actions, like calling an API, querying a database, or performing calculations.

Our first tool will provide a *mock* weather report. This allows us to focus on the agent structure without needing external API keys yet. Later, you could easily swap this mock function with one that calls a real weather service.

**Key Concept: Docstrings are Crucial\!** The agent's LLM relies heavily on the function's **docstring** to understand:

* *What* the tool does.  
* *When* to use it.  
* *What arguments* it requires (`city: str`).  
* *What information* it returns.

**Best Practice:** Write clear, descriptive, and accurate docstrings for your tools. This is essential for the LLM to use the tool correctly.

In [3]:
def get_weather(city: str) -> dict:
    """Retrieves the current weather report for a specified city.

    Args:
        city (str): The name of the city (e.g., "New York", "London", "Tokyo").

    Returns:
        dict: A dictionary containing the weather information.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'report' key with weather details.
              If 'error', includes an 'error_message' key.
    """
    print(f"--- Tool: get_weather called for city: {city} ---") # Log tool execution
    city_normalized = city.lower().replace(" ", "") # Basic normalization

    # Mock weather data
    mock_weather_db = {
        "newyork": {"status": "success", "report": "The weather in New York is sunny with a temperature of 25°C."},
        "london": {"status": "success", "report": "It's cloudy in London with a temperature of 15°C."},
        "tokyo": {"status": "success", "report": "Tokyo is experiencing light rain and a temperature of 18°C."},
    }

    if city_normalized in mock_weather_db:
        return mock_weather_db[city_normalized]
    else:
        return {"status": "error", "error_message": f"Sorry, I don't have weather information for '{city}'."}

# Example tool usage (optional test)
print(get_weather("New York"))
print(get_weather("Paris"))

--- Tool: get_weather called for city: New York ---
{'status': 'success', 'report': 'The weather in New York is sunny with a temperature of 25°C.'}
--- Tool: get_weather called for city: Paris ---
{'status': 'error', 'error_message': "Sorry, I don't have weather information for 'Paris'."}


---

**2\. Define the Agent (`weather_agent`)**

Now, let's create the **Agent** itself. An `Agent` in ADK orchestrates the interaction between the user, the LLM, and the available tools.

We configure it with several key parameters:

* `name`: A unique identifier for this agent (e.g., "weather\_agent\_v1").  
* `model`: Specifies which LLM to use (e.g., `MODEL_GEMINI_2_5_PRO`). We'll start with a specific Gemini model.  
* `description`: A concise summary of the agent's overall purpose. This becomes crucial later when other agents need to decide whether to delegate tasks to *this* agent.  
* `instruction`: Detailed guidance for the LLM on how to behave, its persona, its goals, and specifically *how and when* to utilize its assigned `tools`.  
* `tools`: A list containing the actual Python tool functions the agent is allowed to use (e.g., `[get_weather]`).

**Best Practice:** Provide clear and specific `instruction` prompts. The more detailed the instructions, the better the LLM can understand its role and how to use its tools effectively. Be explicit about error handling if needed.

**Best Practice:** Choose descriptive `name` and `description` values. These are used internally by ADK and are vital for features like automatic delegation (covered later).

In [4]:
# Use one of the model constants defined earlier
AGENT_MODEL = MODEL_GEMINI_2_0_FLASH # Starting with Gemini

weather_agent = Agent(
    name="weather_agent_v1",
    model=AGENT_MODEL, # Can be a string for Gemini
    description="Provides weather information for specific cities.",
    instruction="You are a helpful weather assistant. "
                "When the user asks for the weather in a specific city, "
                "use the 'get_weather' tool to find the information. "
                "If the tool returns an error, inform the user politely. "
                "If the tool is successful, present the weather report clearly.",
    tools=[get_weather], # Pass the function directly
)

print(f"Agent '{weather_agent.name}' created using model '{AGENT_MODEL}'.")

Agent 'weather_agent_v1' created using model 'gemini-2.0-flash'.


---

**3\. Setup Runner and Session Service**

To manage conversations and execute the agent, we need two more components:

* `SessionService`: Responsible for managing conversation history and state for different users and sessions. The `InMemorySessionService` is a simple implementation that stores everything in memory, suitable for testing and simple applications. It keeps track of the messages exchanged. We'll explore state persistence more in Step 4\.  
* `Runner`: The engine that orchestrates the interaction flow. It takes user input, routes it to the appropriate agent, manages calls to the LLM and tools based on the agent's logic, handles session updates via the `SessionService`, and yields events representing the progress of the interaction.

In [5]:
# --- Session Management ---
# Key Concept: SessionService stores conversation history & state.
# InMemorySessionService is simple, non-persistent storage for this tutorial.
session_service = InMemorySessionService()

# Define constants for identifying the interaction context
APP_NAME = "weather_tutorial_app"
USER_ID = "user_1"
SESSION_ID = "session_003" # Using a fixed ID for simplicity

# Create the specific session where the conversation will happen
session = session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# --- Runner ---
# Key Concept: Runner orchestrates the agent execution loop.
runner = Runner(
    agent=weather_agent, # The agent we want to run
    app_name=APP_NAME,   # Associates runs with our app
    session_service=session_service # Uses our session manager
)
print(f"Runner created for agent '{runner.agent.name}'.")

Session created: App='weather_tutorial_app', User='user_1', Session='session_003'
Runner created for agent 'weather_agent_v1'.


---

**4\. Interact with the Agent**

We need a way to send messages to our agent and receive its responses. Since LLM calls and tool executions can take time, ADK's `Runner` operates asynchronously.

We'll define an `async` helper function (`call_agent_async`) that:

1. Takes a user query string.  
2. Packages it into the ADK `Content` format.  
3. Calls `runner.run_async`, providing the user/session context and the new message.  
4. Iterates through the **Events** yielded by the runner. Events represent steps in the agent's execution (e.g., tool call requested, tool result received, intermediate LLM thought, final response).  
5. Identifies and prints the **final response** event using `event.is_final_response()`.

**Why `async`?** Interactions with LLMs and potentially tools (like external APIs) are I/O-bound operations. Using `asyncio` allows the program to handle these operations efficiently without blocking execution.

In [7]:
async def call_agent_async(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n>>> User Query: {query}")

  # Prepare the user's message in ADK format
  content = types.Content(role='user', parts=[types.Part(text=query)])

  final_response_text = "Agent did not produce a final response." # Default

  # Key Concept: run_async executes the agent logic and yields Events.
  # We iterate through events to find the final answer.
  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
      # You can uncomment the line below to see *all* events during execution
      # print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

      # Key Concept: is_final_response() marks the concluding message for the turn.
      if event.is_final_response():
          if event.content and event.content.parts:
             # Assuming text response in the first part
             final_response_text = event.content.parts[0].text
          elif event.actions and event.actions.escalate: # Handle potential errors/escalations
             final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
          # Add more checks here if needed (e.g., specific error codes)
          break # Stop processing events once the final response is found

  print(f"<<< Agent Response: {final_response_text}")

---

**5\. Run the Conversation**

Finally, let's test our setup by sending a few queries to the agent. We wrap our `async` calls in a main `async` function and run it using `await`.

Watch the output:

* See the user queries.  
* Notice the `--- Tool: get_weather called... ---` logs when the agent uses the tool.  
* Observe the agent's final responses, including how it handles the case where weather data isn't available (for Paris).

In [8]:
# We need an async function to await our interaction helper
async def run_conversation():
    await call_agent_async("What is the weather like in London?",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)

    await call_agent_async("How about Paris?",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID) # Expecting the tool's error message

    await call_agent_async("Tell me the weather in New York",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)

# Execute the conversation using await in an async context (like Colab/Jupyter)
await run_conversation()


>>> User Query: What is the weather like in London?


--- Tool: get_weather called for city: London ---
<<< Agent Response: The weather in London is cloudy with a temperature of 15°C.


>>> User Query: How about Paris?


--- Tool: get_weather called for city: Paris ---
<<< Agent Response: Sorry, I don't have weather information for Paris.


>>> User Query: Tell me the weather in New York


--- Tool: get_weather called for city: New York ---
<<< Agent Response: The weather in New York is sunny with a temperature of 25°C.



## Step 2: Expanding Your Agent – Multi-Tool Reasoning (Single Agent, Multiple Tools)

Now that we’ve built a basic agent with a single tool, let’s expand its capabilities. In this step, you’ll enhance your agent so it can do more than just fetch the weather—it will also be able to **recommend events** based on the city the user asks about.

This demonstrates a key power of agents in ADK: **multi-tool reasoning**. The agent will analyze the user’s intent and decide which tool to use—`get_weather` or `get_event_recommendations`.

We’ll still be using mock data so you can focus on architecture, logic, and reasoning.

---
**1\. Add a Second Tool (`get_event_recommendations`)**

Just like `get_weather`, this second tool is a simple Python function with a well-written docstring that the LLM will rely on to understand its purpose and usage.

Remember: **Tools = Capabilities.** By adding tools, we are expanding what the agent *can do*, but it’s still up to the agent to choose the right one.

This new tool provides mock event recommendations for a few cities. It normalizes the input city string and returns either a list of suggested activities or an error message if the city isn’t found in the mock database.

As always, the **docstring is critical**. It helps the agent understand:
- What the tool does.
- When it should be used.
- What input it expects (`city: str`).
- What kind of output it returns (a structured dictionary with events or error messages).


In [9]:
def get_event_recommendations(city: str) -> dict:
    """Provides event recommendations for a specified city based on mock weather conditions.

    Args:
        city (str): The name of the city (e.g., "New York", "London", "Tokyo").

    Returns:
        dict: A dictionary containing event recommendations.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes an 'events' key with a list of strings.
              If 'error', includes an 'error_message' key.
    """
    print(f"--- Tool: get_event_recommendations called for city: {city} ---")
    city_normalized = city.lower().replace(" ", "")

    # Mock weather-dependent event suggestions
    mock_event_db = {
        "newyork": ["Picnic in Central Park", "Outdoor jazz concert at Bryant Park", "Sunset boat cruise"],
        "london": ["Visit the British Museum", "Tea tasting in Soho", "Explore Borough Market"],
        "tokyo": ["Sumida Aquarium visit", "Shinjuku shopping tour", "Soba noodle workshop"],
    }

    if city_normalized in mock_event_db:
        return {"status": "success", "events": mock_event_db[city_normalized]}
    else:
        return {"status": "error", "error_message": f"Sorry, I don't have event recommendations for '{city}'."}

# Example tool usage (optional test)
print(get_event_recommendations("New York"))
print(get_event_recommendations("Paris"))

--- Tool: get_event_recommendations called for city: New York ---
{'status': 'success', 'events': ['Picnic in Central Park', 'Outdoor jazz concert at Bryant Park', 'Sunset boat cruise']}
--- Tool: get_event_recommendations called for city: Paris ---
{'status': 'error', 'error_message': "Sorry, I don't have event recommendations for 'Paris'."}


---

**2\. Define the Agent (`weather_recommendations_agent_v1`)**

It is very similar to next phase but the agent with a different instruction on when to use which tools logic and reaosn.

In [10]:
# Use one of the model constants defined earlier
AGENT_MODEL = MODEL_GEMINI_2_0_FLASH # Starting with Gemini

weather_agent = Agent(
    name="weather_recommendations_agent_v1",
    model=AGENT_MODEL, # Can be a string for Gemini
    description = "Provides weather and local event recommendation information for specific cities.",
    instruction = "You are a helpful travel assistant. "
                  "When the user asks for the weather in a specific city, use the 'get_weather' tool to retrieve the weather report. "
                  "When the user asks about local events, use the 'get_event_recommendations' tool to suggest activities. "
                  "If either tool returns an error, politely inform the user. "
                  "If successful, clearly present the weather or event recommendations. "
                  "You may also suggest events that fit the current weather if both are requested.",
    tools=[get_weather, get_event_recommendations], # Pass the function directly
)

print(f"Agent '{weather_agent.name}' created using model '{AGENT_MODEL}'.")

Agent 'weather_recommendations_agent_v1' created using model 'gemini-2.0-flash'.


---

**3\. Setup Runner and Session Service**

The same as previouse step. 


In [11]:
# --- Session Management ---
# Key Concept: SessionService stores conversation history & state.
# InMemorySessionService is simple, non-persistent storage for this tutorial.
session_service = InMemorySessionService()

# Define constants for identifying the interaction context
APP_NAME = "travel_info_tutorial_app"
USER_ID = "user_2"
SESSION_ID = "session_002" # Using a fixed ID for simplicity

# Create the specific session where the conversation will happen
session = session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# --- Runner ---
# Key Concept: Runner orchestrates the agent execution loop.
runner = Runner(
    agent=weather_agent, # The agent we want to run
    app_name=APP_NAME,   # Associates runs with our app
    session_service=session_service # Uses our session manager
)
print(f"Runner created for agent '{runner.agent.name}'.")

Session created: App='travel_info_tutorial_app', User='user_2', Session='session_002'
Runner created for agent 'weather_recommendations_agent_v1'.


---

**4\. Interact with the Agent**

The same as previouse step. 


In [12]:
async def call_agent_async(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n>>> User Query: {query}")

  # Prepare the user's message in ADK format
  content = types.Content(role='user', parts=[types.Part(text=query)])

  final_response_text = "Agent did not produce a final response." # Default

  # Key Concept: run_async executes the agent logic and yields Events.
  # We iterate through events to find the final answer.
  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
      # You can uncomment the line below to see *all* events during execution
      # print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

      # Key Concept: is_final_response() marks the concluding message for the turn.
      if event.is_final_response():
          if event.content and event.content.parts:
             # Assuming text response in the first part
             final_response_text = event.content.parts[0].text
          elif event.actions and event.actions.escalate: # Handle potential errors/escalations
             final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
          # Add more checks here if needed (e.g., specific error codes)
          break # Stop processing events once the final response is found

  print(f"<<< Agent Response: {final_response_text}")

**5\. Run the Conversation**

The same as previouse step. 


In [13]:
# We need an async function to await our interaction helper
async def run_conversation():
    await call_agent_async("What is the weather like in London?",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)
  
    await call_agent_async("What can I do there?",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)

# Execute the conversation using await in an async context (like Colab/Jupyter)
await run_conversation()


>>> User Query: What is the weather like in London?


--- Tool: get_weather called for city: London ---
<<< Agent Response: It's cloudy in London with a temperature of 15°C.


>>> User Query: What can I do there?


--- Tool: get_event_recommendations called for city: London ---
<<< Agent Response: Here are some events you might enjoy in London: Visit the British Museum, tea tasting in Soho, or explore Borough Market.



## Step 3: Building an Agent Team – Delegation + Multi-Tool Usage

In Steps 1 and 2, we built a single agent focused on weather lookups. While effective for that specific task, real-world applications often require handling a broader range of user interactions. Although we could continue adding more tools and logic to a single agent, this approach quickly becomes complex and less maintainable.

### Why Build an Agent Team?

Creating a team of specialized agents with clear responsibilities offers several advantages:

- **Modularity**: Easier to develop, test, and maintain individual agents.
- **Specialization**: Each agent can be fine-tuned for its specific task (e.g., greetings vs. weather).
- **Scalability**: New capabilities can be added by introducing new tools or sub-agents.
- **Efficiency**: Simpler tasks can be delegated to lightweight models, reducing resource usage.

### What We’ll Do in This Step

- Define tools for:
  - `get_weather` – retrieving weather information.
  - `get_event_recommendations` – suggesting local events and activities.
  - `say_hello` – responding to greetings.
  - `say_goodbye` – responding to farewells.
- Create two specialized sub-agents:
  - `greeting_agent` – handles greetings.
  - `farewell_agent` – handles farewells.
- Update our main agent (`weather_agent_v2`) to act as a **root agent** with multiple capabilities:
  - Use `get_weather` and `get_event_recommendations` directly, depending on the user's request.
  - Delegate to `greeting_agent` or `farewell_agent` when the user intent matches.

### Goal

By the end of this step, you'll have a multi-agent, multi-tool setup that:

- Handles **weather** and **event** requests directly.
- Delegates **greetings** and **farewells** to specialized sub-agents.
- Uses intent detection to route user queries to the right capability.


---

**1\. Define Tools for Sub-Agents**

First, let's create the simple Python functions that will serve as tools for our new specialist agents. Remember, clear docstrings are vital for the agents that will use them.

In [14]:
# Ensure 'get_weather' from Step 1 is available if running this step independently.
# def get_weather(city: str) -> dict: ... (from Step 1)

def say_hello(name: str = "there") -> str:
    """Provides a simple greeting, optionally addressing the user by name.

    Args:
        name (str, optional): The name of the person to greet. Defaults to "there".

    Returns:
        str: A friendly greeting message.
    """
    print(f"--- Tool: say_hello called with name: {name} ---")
    return f"Hello, {name}!"

def say_goodbye() -> str:
    """Provides a simple farewell message to conclude the conversation."""
    print(f"--- Tool: say_goodbye called ---")
    return "Goodbye! Have a great day."

print("Greeting and Farewell tools defined.")

# Optional self-test
print(say_hello("Alice"))
print(say_goodbye())

Greeting and Farewell tools defined.
--- Tool: say_hello called with name: Alice ---
Hello, Alice!
--- Tool: say_goodbye called ---
Goodbye! Have a great day.


---

**2\. Define the Sub-Agents (Greeting & Farewell)**

Now, create the `Agent` instances for our specialists. Notice their highly focused `instruction` and, critically, their clear `description`. The `description` is the primary information the *root agent* uses to decide *when* to delegate to these sub-agents.

We can even use different LLMs for these sub-agents\! Let's assign GPT-4o to the Greeting Agent and keep the Farewell Agent using GPT-4o as well (you could easily switch one to Claude or Gemini if desired and API keys are set).

**Best Practice:** Sub-agent `description` fields should accurately and concisely summarize their specific capability. This is crucial for effective automatic delegation.

**Best Practice:** Sub-agent `instruction` fields should be tailored to their limited scope, telling them exactly what to do and *what not* to do (e.g., "Your *only* task is...").

In [15]:
# --- Greeting Agent ---
greeting_agent = None
try:
    greeting_agent = Agent(
        # Using a potentially different/cheaper model for a simple task
        #model=LiteLlm(model=MODEL_GEMINI_2_0_FLASH),
        model=MODEL_GEMINI_2_0_FLASH,
        name="greeting_agent",
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting to the user. "
                    "Use the 'say_hello' tool to generate the greeting. "
                    "If the user provides their name, make sure to pass it to the tool. "
                    "Do not engage in any other conversation or tasks.",
        description="Handles simple greetings and hellos using the 'say_hello' tool.", # Crucial for delegation
        tools=[say_hello],
    )
    print(f"✅ Agent '{greeting_agent.name}' created using model '{MODEL_GEMINI_2_0_FLASH}'.")
except Exception as e:
    print(f"❌ Could not create Greeting agent. Check API Key ({MODEL_GEMINI_2_0_FLASH}). Error: {e}")

# --- Farewell Agent ---
farewell_agent = None
try:
    farewell_agent = Agent(
        # Can use the same or a different model
        #model=LiteLlm(model=MODEL_GEMINI_2_0_FLASH), # Sticking with GPT for this example
        model=MODEL_GEMINI_2_0_FLASH,
        name="farewell_agent",
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message. "
                    "Use the 'say_goodbye' tool when the user indicates they are leaving or ending the conversation "
                    "(e.g., using words like 'bye', 'goodbye', 'thanks bye', 'see you'). "
                    "Do not perform any other actions.",
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.", # Crucial for delegation
        tools=[say_goodbye],
    )
    print(f"✅ Agent '{farewell_agent.name}' created using model '{MODEL_GEMINI_2_0_FLASH}'.")
except Exception as e:
    print(f"❌ Could not create Farewell agent. Check API Key ({MODEL_GEMINI_2_0_FLASH}). Error: {e}")

✅ Agent 'greeting_agent' created using model 'gemini-2.0-flash'.
✅ Agent 'farewell_agent' created using model 'gemini-2.0-flash'.


---

**3\. Define the Root Agent (Weather Agent v2) with Sub-Agents**

Now, we upgrade our `weather_agent`. The key changes are:

* Adding the `sub_agents` parameter: We pass a list containing the `greeting_agent` and `farewell_agent` instances we just created.  
* Updating the `instruction`: We explicitly tell the root agent *about* its sub-agents and *when* it should delegate tasks to them.

**Key Concept: Automatic Delegation (Auto Flow)** By providing the `sub_agents` list, ADK enables automatic delegation. When the root agent receives a user query, its LLM considers not only its own instructions and tools but also the `description` of each sub-agent. If the LLM determines that a query aligns better with a sub-agent's described capability (e.g., "Handles simple greetings"), it will automatically generate a special internal action to *transfer control* to that sub-agent for that turn. The sub-agent then processes the query using its own model, instructions, and tools.

**Best Practice:** Ensure the root agent's instructions clearly guide its delegation decisions. Mention the sub-agents by name and describe the conditions under which delegation should occur.

In [16]:
# Ensure sub-agents were created successfully before defining the root agent.
# Also ensure the original 'get_weather' tool is defined.
root_agent = None
runner_root = None # Initialize runner

if greeting_agent and farewell_agent and 'get_weather' and 'get_event_recommendations' in globals():
    # Let's use a capable Gemini model for the root agent to handle orchestration
    root_agent_model = MODEL_GEMINI_2_0_FLASH

    weather_agent_team = Agent(
        name="weather_agent_v2", # Give it a new version name
        model=root_agent_model,
        description="The main coordinator agent. Handles weather requestsm recommend event and delegates greetings/farewells to specialists.",
        instruction="You are a helpful travel assistant coordinating a team of specialized sub-agents. Your main responsibility is to assist users with weather and local event information."
                    "Use the get_weather tool only when the user asks for specific weather information (e.g., “What’s the weather in Paris?”)."
                    "Use the get_event_recommendations tool when the user inquires about local activities or events."
                    "If both weather and events are requested, you may suggest events that align with the current weather conditions."
                    "If either tool returns an error, politely inform the user. If successful, present the results clearly."
                    "Delegate greetings like “Hi” or “Hello” to the greeting_agent."
                    "Delegate farewells like “Bye” or “See you” to the farewell_agent."
                    "if the user’s request does not match any of these categories, respond appropriately or state that you cannot handle it.",
        #tools=[get_weather], # Root agent still needs the weather tool for its core task
        tools=[get_weather, get_event_recommendations], # Pass the function directly
        # Key change: Link the sub-agents here!
        sub_agents=[greeting_agent, farewell_agent]
    )
    print(f"✅ Root Agent '{weather_agent_team.name}' created using model '{root_agent_model}' with sub-agents: {[sa.name for sa in weather_agent_team.sub_agents]}")

else:
    print("❌ Cannot create root agent because one or more sub-agents failed to initialize or 'get_weather' tool is missing.")
    if not greeting_agent: print(" - Greeting Agent is missing.")
    if not farewell_agent: print(" - Farewell Agent is missing.")
    if 'get_weather' not in globals(): print(" - get_weather function is missing.")

✅ Root Agent 'weather_agent_v2' created using model 'gemini-2.0-flash' with sub-agents: ['greeting_agent', 'farewell_agent']


**4\. Interact with the Agent Team**

Now that we've defined our root agent (`weather_agent_team` - *Note: Ensure this variable name matches the one defined in the previous code block, likely `# @title Define the Root Agent with Sub-Agents`, which might have named it `root_agent`*) with its specialized sub-agents, let's test the delegation mechanism.

The following code block will:

1.  Define an `async` function `run_team_conversation`.
2.  Inside this function, create a *new, dedicated* `InMemorySessionService` and a specific session (`session_001_agent_team`) just for this test run. This isolates the conversation history for testing the team dynamics.
3.  Create a `Runner` (`runner_agent_team`) configured to use our `weather_agent_team` (the root agent) and the dedicated session service.
4.  Use our updated `call_agent_async` function to send different types of queries (greeting, weather request, farewell) to the `runner_agent_team`. We explicitly pass the runner, user ID, and session ID for this specific test.
5.  Immediately execute the `run_team_conversation` function.

We expect the following flow:

1.  The "Hello there!" query goes to `runner_agent_team`.
2.  The root agent (`weather_agent_team`) receives it and, based on its instructions and the `greeting_agent`'s description, delegates the task.
3.  `greeting_agent` handles the query, calls its `say_hello` tool, and generates the response.
4.  The "What is the weather in New York?" query is *not* delegated and is handled directly by the root agent using its `get_weather` tool.
5.  The "Thanks, bye!" query is delegated to the `farewell_agent`, which uses its `say_goodbye` tool.



In [17]:
# Ensure the root agent (e.g., 'weather_agent_team' or 'root_agent' from the previous cell) is defined.
# Ensure the call_agent_async function is defined.

# Check if the root agent variable exists before defining the conversation function
root_agent_var_name = 'root_agent' # Default name from Step 3 guide
if 'weather_agent_team' in globals(): # Check if user used this name instead
    root_agent_var_name = 'weather_agent_team'
elif 'root_agent' not in globals():
    print("⚠️ Root agent ('root_agent' or 'weather_agent_team') not found. Cannot define run_team_conversation.")
    # Assign a dummy value to prevent NameError later if the code block runs anyway
    root_agent = None

if root_agent_var_name in globals() and globals()[root_agent_var_name]:
    async def run_team_conversation():
        print("\n--- Testing Agent Team Delegation ---")
        # InMemorySessionService is simple, non-persistent storage for this tutorial.
        session_service = InMemorySessionService()

        # Define constants for identifying the interaction context
        APP_NAME = "weather_tutorial_agent_team"
        USER_ID = "user_1_agent_team"
        SESSION_ID = "session_001_agent_team" # Using a fixed ID for simplicity

        # Create the specific session where the conversation will happen
        session = session_service.create_session(
            app_name=APP_NAME,
            user_id=USER_ID,
            session_id=SESSION_ID
        )
        print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

        # --- Get the actual root agent object ---
        # Use the determined variable name
        actual_root_agent = globals()[root_agent_var_name]

        # Create a runner specific to this agent team test
        runner_agent_team = Runner(
            agent=actual_root_agent, # Use the root agent object
            app_name=APP_NAME,       # Use the specific app name
            session_service=session_service # Use the specific session service
            )
        # Corrected print statement to show the actual root agent's name
        print(f"Runner created for agent '{actual_root_agent.name}'.")

        # Always interact via the root agent's runner, passing the correct IDs
        await call_agent_async(query = "Hello there! this is Amir",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)
        await call_agent_async(query = "What is the weather in London?",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)
        await call_agent_async(query = "What can I do there?",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)
        await call_agent_async(query = "Thanks, bye!",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)

    # Execute the conversation
    # Note: This may require API keys for the models used by root and sub-agents!
    await run_team_conversation()
else:
    print("\n⚠️ Skipping agent team conversation as the root agent was not successfully defined in the previous step.")



--- Testing Agent Team Delegation ---
Session created: App='weather_tutorial_agent_team', User='user_1_agent_team', Session='session_001_agent_team'
Runner created for agent 'weather_agent_v2'.

>>> User Query: Hello there! this is Amir


--- Tool: say_hello called with name: Amir ---
<<< Agent Response: Hello, Amir!


>>> User Query: What is the weather in London?


--- Tool: get_weather called for city: London ---
<<< Agent Response: It's cloudy in London with a temperature of 15°C.


>>> User Query: What can I do there?


--- Tool: get_event_recommendations called for city: London ---
<<< Agent Response: I recommend the following events in London: Visit the British Museum, Tea tasting in Soho, and Explore Borough Market.


>>> User Query: Thanks, bye!


--- Tool: say_goodbye called ---
<<< Agent Response: Goodbye! Have a great day.

